# Tail-Risk and Margin Procyclicality Lab

This lab compares:

- **Scenario regimes**: Gaussian baseline vs Student-t fat-tail vs jump-diffusion proxy stress.
- **Margin regimes**: Gaussian VaR vs Expected Shortfall vs stressed VaR.

For each comparison, we generate reproducible report bundles in `reports/` containing:

- Markdown report
- Scenario comparison CSV
- Raw scenario comparison JSON summary

Educational/research use only. Not a production CCP model.

## 1) Setup

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
REPORTS_DIR = REPO_ROOT / "reports"
CONFIG_DIR = REPORTS_DIR / "tail_margin_lab_configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ENV = dict(os.environ)
ENV["PYTHONPATH"] = f"{SRC_DIR}:{ENV.get('PYTHONPATH', '')}" if ENV.get("PYTHONPATH") else str(SRC_DIR)

print("Repo:", REPO_ROOT)
print("Config dir:", CONFIG_DIR)

## 2) Shared Base Configuration Builder

In [ ]:
def base_config(
    *,
    run_name: str,
    margin_method: str,
    scenario_name: str,
    stress_label: str,
    volatility: float,
    liquidity_spread_multiplier: float,
    jump_probability: float = 0.0,
    jump_size_distribution: str = "none",
    include_closeout: bool = False,
    anti_procyclicality_buffer: float = 0.0,
):
    return {
        "run_name": run_name,
        "time_horizon_days": 252,
        "include_liquidity_closeout_cost": include_closeout,
        "margin": {
            "method": margin_method,
            "confidence_level": 0.99,
            "margin_period_of_risk_days": 5,
            "stressed_var_vol_multiplier": 1.6,
            "anti_procyclicality_buffer": anti_procyclicality_buffer,
            "volatility_floor": 0.10,
            "concentration_addon": 0.03,
            "liquidity_addon": 0.03,
        },
        "waterfall": {
            "ccp_skin_in_game": 100000000,
            "assessment_cap_multiple": 0.20,
            "include_defaulter_df": True,
            "default_fund_method": "cover2",
            "default_fund_value": 0.0,
        },
        "closeout": {
            "base_spread_cost": 0.01,
            "volatility_liquidity_multiplier": 0.02,
            "concentration_penalty": 0.05,
            "market_depth_penalty": 0.01,
        },
        "contagion": {"liquidity_breach_ratio": 1.0},
        "scenarios": [
            {
                "scenario_name": scenario_name,
                "stress_label": stress_label,
                "volatility": volatility,
                "liquidity_spread_multiplier": liquidity_spread_multiplier,
                "jump_probability": jump_probability,
                "jump_size_distribution": jump_size_distribution,
            }
        ],
        "members": [
            {
                "member_id": "CM_A",
                "default_fund_contribution": 150000,
                "liquidity_buffer": 300000,
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 1400000, "price": 1.0}]},
            },
            {
                "member_id": "CM_B",
                "default_fund_contribution": 110000,
                "liquidity_buffer": 240000,
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 1000000, "price": 1.0}]},
            },
            {
                "member_id": "CM_C",
                "default_fund_contribution": 90000,
                "liquidity_buffer": 200000,
                "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 800000, "price": 1.0}]},
            },
        ],
    }

## 3) Build Experiment Configs

We create two groups:

1. **Scenario comparisons** (fixed margin method):
   - Gaussian baseline vs Student-t proxy stress
   - Gaussian baseline vs jump-diffusion proxy stress

2. **Margin regime comparisons** (fixed stress scenario):
   - Gaussian VaR vs Expected Shortfall
   - Gaussian VaR vs stressed VaR

In [ ]:
configs = {}

# Scenario-focused configs (fixed margin method)
configs["gaussian_var_baseline"] = base_config(
    run_name="gaussian_var_baseline",
    margin_method="gaussian_var",
    scenario_name="gaussian_baseline",
    stress_label="baseline",
    volatility=0.20,
    liquidity_spread_multiplier=1.00,
    include_closeout=False,
)

configs["gaussian_var_fat_tail"] = base_config(
    run_name="gaussian_var_fat_tail",
    margin_method="gaussian_var",
    scenario_name="student_t_fat_tail",
    stress_label="fat_tail",
    volatility=0.30,
    liquidity_spread_multiplier=1.30,
    jump_probability=0.0,
    jump_size_distribution="student_t_df4",
    include_closeout=True,
)

configs["gaussian_var_jump"] = base_config(
    run_name="gaussian_var_jump",
    margin_method="gaussian_var",
    scenario_name="jump_diffusion_stress",
    stress_label="jump_diffusion",
    volatility=0.28,
    liquidity_spread_multiplier=1.25,
    jump_probability=0.03,
    jump_size_distribution="normal(mean=-0.10,std=0.04)",
    include_closeout=True,
)

# Margin-regime configs (fixed stressed scenario)
configs["es_stressed"] = base_config(
    run_name="es_stressed",
    margin_method="expected_shortfall",
    scenario_name="high_vol_stress",
    stress_label="stressed",
    volatility=0.35,
    liquidity_spread_multiplier=1.40,
    include_closeout=True,
)

configs["stressed_var_stressed"] = base_config(
    run_name="stressed_var_stressed",
    margin_method="stressed_var",
    scenario_name="high_vol_stress",
    stress_label="stressed",
    volatility=0.35,
    liquidity_spread_multiplier=1.40,
    include_closeout=True,
    anti_procyclicality_buffer=0.05,
)

paths = {}
for name, payload in configs.items():
    path = CONFIG_DIR / f"{name}.json"
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    paths[name] = path

print("Wrote config files:")
for k, v in paths.items():
    print(f"- {k}: {v}")

## 4) Helper: Run a Comparison Bundle

In [ ]:
def run_bundle(stressed_config: Path, baseline_config: Path, bundle_name: str) -> Path:
    bundle_dir = REPORTS_DIR / bundle_name
    cmd = [
        sys.executable,
        "-m",
        "clearrisk.cli",
        "report",
        "--config",
        str(stressed_config),
        "--compare-config",
        str(baseline_config),
        "--bundle-dir",
        str(bundle_dir),
    ]
    completed = subprocess.run(cmd, env=ENV, check=True, capture_output=True, text=True)
    print(completed.stdout)
    return bundle_dir

## 5) Run Lab Bundles

This produces three reproducible bundles:

- `reports/lab_gaussian_vs_fat_tail`
- `reports/lab_gaussian_vs_jump`
- `reports/lab_var_vs_es`
- `reports/lab_var_vs_stressed_var`

In [ ]:
bundles = {}
bundles["gaussian_vs_fat_tail"] = run_bundle(
    paths["gaussian_var_fat_tail"],
    paths["gaussian_var_baseline"],
    "lab_gaussian_vs_fat_tail",
)

bundles["gaussian_vs_jump"] = run_bundle(
    paths["gaussian_var_jump"],
    paths["gaussian_var_baseline"],
    "lab_gaussian_vs_jump",
)

bundles["var_vs_es"] = run_bundle(
    paths["es_stressed"],
    paths["gaussian_var_baseline"],
    "lab_var_vs_es",
)

bundles["var_vs_stressed_var"] = run_bundle(
    paths["stressed_var_stressed"],
    paths["gaussian_var_baseline"],
    "lab_var_vs_stressed_var",
)

print("Generated bundles:")
for name, path in bundles.items():
    print(f"- {name}: {path}")

## 6) Collect Comparison Highlights

In [ ]:
def read_summary(bundle_dir: Path):
    summary_path = bundle_dir / "scenario_comparison_summary.json"
    return json.loads(summary_path.read_text(encoding="utf-8"))

highlights = []
for name, bundle_dir in bundles.items():
    s = read_summary(bundle_dir)
    c = s.get("comparison", {})
    highlights.append(
        {
            "experiment": name,
            "tail_loss_ratio_es": c.get("tail_loss_ratio_es", 0.0),
            "margin_adequacy_delta": c.get("margin_adequacy_delta", 0.0),
            "cover2_adequacy_delta": c.get("cover2_adequacy_delta", 0.0),
        }
    )

highlights

In [ ]:
import pandas as pd

df = pd.DataFrame(highlights)
df

## 7) Read One Bundle’s Practitioner CSV

In [ ]:
csv_path = bundles["gaussian_vs_fat_tail"] / "scenario_comparison.csv"
pd.read_csv(csv_path).head(10)

## 8) Notes for Interpretation

- `tail_loss_ratio_es > 1` means stressed ES exceeds baseline ES.
- Negative `margin_adequacy_delta` indicates weaker adequacy under stress relative to baseline.
- Negative `cover2_adequacy_delta` indicates Cover-2 headroom deterioration under stress.

This lab is intentionally synthetic and educational, designed to surface risk tradeoffs rather than forecast real default outcomes.